# Developing data simulation

The objective of this notebook is helping to develop step by step the simulation from `R` into `python`

The first objective will be the creation of a simulation possibility based only on continuous features. No categorical will be considered.

In [1]:
import torch

In [2]:
means = torch.tensor([3.5,-3.5], dtype=torch.float64)
covs = torch.tensor([[1,-0.5],[-0.5,1]], dtype=torch.float64)
mvn = torch.distributions.MultivariateNormal(means, covariance_matrix=covs)

In a 2 dimensional multivariate distribution, we have a vector $X$ of two RV
$$
X = \begin{pmatrix} X_1 \\ X_2 \end{pmatrix}
$$
Mean and VCOV matrix are given by
$$
\mu = \begin{pmatrix} 3.5 \\ -3.5 \end{pmatrix}, \; 
\Sigma = \begin{pmatrix}
1 & -\frac{1}{2} \\
-\frac{1}{2} & 1
\end{pmatrix}
$$

This means when sampling we will only get positive numbers for $X_1$ realizations and negative numbers for $X_2$ realizations

In [3]:
sample_2d = mvn.sample((3,))
print(sample_2d.shape)
sample_2d

torch.Size([3, 2])


tensor([[ 4.0739, -2.5579],
        [ 3.5667, -2.2065],
        [ 4.5279, -4.0313]], dtype=torch.float64)

So in a sample size of `(n,)`, the shape will be `[n, k]` with `k` the number of random variables composing $X$.

With $x^{(r)}_j$ the $j$-th sampled realization of $X_r$, the output looks like
$$
\texttt{sample} = \begin{pmatrix}
    x^{(1)}_1 & \cdots & x^{(k)}_1 \\
    \vdots & \ddots & \vdots \\
    x^{(1)}_n & \cdots & x^{(k)}_n
\end{pmatrix} \in \mathbb{R}^{n \times k}
$$

## Notes on the R-Algorithm

### Calls:

First call is made for the init_population, by

```r
res <- genCreditData(
  #################################### DIMENSIONALITY
  n                = init_sample,  # - sample size = 100
  bad_ratio        = bad_ratio,    # - BAD ratio (if all D = 0) = 0.7
  k_con            = num_feats,    # - no. continuous features = 2
  k_cat            = 0,            # - no. categorical features
  k_bin            = 0,            # - no. binary features
  k_noise          = num_noise,    # - no. white-noise features = 0
  #################################### CONTINUOUS FEATURES
  con_nonlinear    = 0.0,       # - share of nonlinear transformations
  con_mean_bad_dif = mean_dif,  # - mean difference between classes = c(2, 1)
  con_var_bad_dif  = var_dif,   # - share of var/covar difference between classes = 0.5
  con_noise_var    = noise_var, # - variance of noise = 0
  covars           = covars,    # - variance-covariance matrices = list(matrix(c(1,  0.2,  0.2, 1), nrow = 2), matrix(c(1, -0.2, -0.2, 1), nrow = 2))
  #################################### MIXTURE OF GAUSSIANS
  mixture      = mixture,       # - mixture of two Gaussians = FALSE
  mix_mean_dif = mix_mean_dif,  # - mean difference between components  = 5
  mix_var_dif  = mix_var_dif,   # - share of var/covar difference between components = 0
  #################################### OTHER PARAMETERS
  seed             = seed,   # - random seed
  verbose          = F,      # - displaying feedback
  encode_factors   = F)      # - encoding of categorical features
```

`matrix(c(1,  0.2,  0.2, 1)` liefert
$$
\begin{pmatrix}
    1 & 0.2 \\
    0.2 & 1
\end{pmatrix}
$$

The next call is done this way:
```r
new_applicants <- genCreditData(n = sample_size, replicate = res, seed = seed + g)$data
```

Which is a quick way to replicate the arguments of the last call, based on this object (all names are set as objects - which are "`names`" in R)
```r
list(n                = n,
     k_cat            = k_cat,
     k_bin            = k_bin,
     k_noise          = k_noise,
     bad_ratio        = bad_ratio,
     con_mean_bad_dif = con_mean_bad_dif,
     con_var_bad_dif  = con_var_bad_dif,
     con_nonlinear    = con_nonlinear,
     con_noise_var    = con_noise_var,
     mixture          = mixture,
     mix_mean_dif     = mix_mean_dif,
     mix_var_dif      = mix_var_dif,
     cat_levels       = cat_levels,
     cat_var_share    = cat_var_share,
     cat_nonlinear    = cat_nonlinear,
     cat_noise_var    = cat_noise_var,
     bin_prob         = bin_prob,
     bin_mean_bad_dif = bin_mean_bad_dif,
     bin_bad_ratio    = bin_bad_ratio,
     bin_mean_con_dif = bin_mean_con_dif,
     bin_var_bad_dif  = bin_var_bad_dif,
     bin_noise_var    = bin_noise_var,
     encode_factors   = encode_factors,
     verbose          = verbose,
     seed             = seed)
```

however n and seed are not "taken over" - the rest they are

### Relevant parts of the algorithm

1. Set `combo_bad_ratio <- 0.7` and `combo_count <- n`
2. Compute "good" and "bad" sample sizes, as $n \cdot 0.7$ und $n \cdot (1-0.7)$ correspondingly. Ties are solved by 50/50 chance of doing good = n - bad or bad = n - good.
3. Set `mu_1 = c(0,0)` and therefore `mu_2 = c(1,2)=con_mean_bad_dif` (see line 249)

In [239]:
from typing import List, Optional, Tuple, Union
import torch
def random_vcov_matrix(
        k: int,
        generator: Optional[torch.Generator] = None,
        var_range: Tuple[float, float] = (0.0, 1.0),
        prefer_normal_base_sampling: bool = True,
        device: torch.device = torch.get_default_device(),
        dtype: torch.dtype = torch.get_default_dtype(),
        eps: float = 1e-6
    ) -> torch.Tensor:
    """Generate a random positive definite covariance matrix.

    The covariance matrix is produced by:

    1. Generating a base sampling of a matrix :math:`A` (normal or uniform).
    2. Creating a correlation matrix via cosine similarity between the row vectors
       of the base sampling:

       .. math::

          C = (c_{i,j}) = \\left( \\frac{\\langle A_i, A_j \\rangle}{\\|A_i\\| \\|A_j\\|} \\right)

    3. Sampling a variance vector uniformly in ``var_range``, which is used to
       rescale the correlation matrix.
    4. Ensuring positive definiteness via addition of a small diagonal
       perturbation ``eps``.
    5. Make sure exact symmetry, so that rounding point instability does not
       lead to unsymmetric results. 

    Args:
        k (int): Dimension of the covariance matrix.
        generator (torch.Generator, optional): Random number generator for reproducibility.
        var_range (Tuple[float, float], optional): Range for diagonal variances. Defaults to (0.0, 1.0).
        prefer_normal_base_sampling (bool, optional): If True, use normal distribution for base sampling.
            If False, use uniform distribution. Defaults to True.
        device (torch.device, optional): Device on which to allocate the tensor.
            Defaults to ``torch.get_default_device()``.
        dtype (torch.dtype, optional): Data type of the returned tensor.
            Defaults to ``torch.get_default_dtype()``.
        eps (float, optional): Small positive value added to the diagonal to ensure positive definiteness.
            Defaults to 1e-6.

    Returns:
        torch.Tensor: A symmetric, positive definite covariance matrix of shape ``(k, k)``.

    Raises:
        ValueError: If ``var_range`` is not a valid (min, max) tuple.

    Example:
        >>> g = torch.Generator().manual_seed(42)
        >>> cov = random_vcov_matrix(4, generator=g)
        >>> cov.shape
        torch.Size([4, 4])
    """

    # Step 1: Generate base sampling
    if prefer_normal_base_sampling:
        A = torch.randn((k, k), generator=generator, dtype=dtype, device=device) # random normal matrix, sparser correlations for high k
    else:
        A = 2*torch.rand((k,k), generator=generator, dtype = dtype, device=device) - 1 # random uniform matrix, correlations closer to 0, the higher k

    # Step 2: Define correlation matrix from base sampling
    Q = A @ A.T # Make sure of symmetry while using full randomness
    D = torch.sqrt(torch.diag(Q)) # Help vector for normalization
    corr_mat = Q / torch.outer(D, D) # corr_mat[i, j] = cosine_similarity(A[i], A[j]), so range [-1, 1] guaranteed

    # Step 3: Rescale corr_mat with sampled variances
    ## Variance sampling from uniform distribution
    variances = torch.rand(k, generator=generator, dtype = dtype, device=device) * (var_range[1] - var_range[0]) + var_range[0]
    ## Rescaling via outer prouct of standard deviations
    stds = torch.sqrt(variances)
    norm_factors_pearson_corr = torch.outer(stds, stds) # guaranteed to be symmetric, denominators of pearson correlation
    vcov = corr_mat * norm_factors_pearson_corr

    # Step 4: Avoid semi positive definitness of the matrix
    vcov = vcov + eps * torch.eye(k, device=device, dtype=dtype)

    # Step 5: Ensure **exact** symmetry without compromising randomness
    i, j = torch.tril_indices(k, k, offset=-1)
    vcov[i, j] = vcov[j, i]

    
    return vcov

def eigen_decomp_proj_to_pd(
    mat: torch.Tensor,
    eps: float = 1e-6,
    ensure_symmetry: bool = False
) -> torch.Tensor:
    """Project a matrix onto the positive definite (PD) cone via eigen-decomposition.

    The procedure ensures the output is symmetric and positive semidefinite by:
    
    1. Optionally symmetrizing the input matrix.
    2. Performing eigen-decomposition.
    3. Clipping eigenvalues below ``eps`` to enforce non-negativity.
    4. Reconstructing the matrix from clipped eigenvalues and eigenvectors.
    5. Symmetrizing the result again to avoid numerical drift.

    Args:
        mat (torch.Tensor): Input square matrix of shape ``(k, k)``.
        eps (float, optional): Minimum eigenvalue threshold to enforce positive definiteness.
            Defaults to ``1e-6``.
        ensure_symmetry (bool, optional): If True, symmetrize the input before decomposition.
            Defaults to False.

    Returns:
        torch.Tensor: Symmetric positive semidefinite matrix of shape ``(k, k)``.

    Example:
        >>> M = torch.tensor([[1.0, 2.0], [2.0, -3.0]])
        >>> M_psd = eigen_decomp_proj_to_pd(M)
        >>> torch.linalg.eigvalsh(M_psd)
        tensor([1.0133e-06, 1.8284e+00])
    """
    # Ensure symmetry
    if ensure_symmetry:
        mat = (mat + mat.T) / 2
    
    # Eigen-decomposition
    eigvals, eigvecs = torch.linalg.eigh(mat)
    
    # Clip eigenvalues to non-negative
    eigvals_clipped = torch.clamp(eigvals, min=eps)
    
    # Reconstruct
    mat_psd = eigvecs @ torch.diag(eigvals_clipped) @ eigvecs.T
    
    # Ensure symmetry again
    return (mat_psd + mat_psd.T) / 2

def generate_sigma_bad_and_good(
    k: int,
    proportion_var_dif: float,
    generator: torch.Generator,
    var_range: Tuple[float, float] = (0.0, 1.0),
    eps: float = 1e-6,
    device: torch.device = torch.device("cpu"),
    dtype: torch.dtype = torch.float64
) -> Tuple[torch.Tensor, torch.Tensor]:
    """Generate a pair of covariance matrices: one 'good' baseline and one 'bad' perturbed version.

    The construction proceeds as follows:

    1. Generate two baseline covariance matrices using ``random_vcov_matrix``.
    2. Sample a random mask over the upper-triangular entries (including diagonal).
    3. Copy selected entries from the 'good' matrix into the 'bad' matrix, leaving
       others perturbed.
    4. Reflect the upper-triangular entries to the lower-triangular part to ensure symmetry.
    5. Project the 'bad' matrix onto the positive definite cone using
       :func:`eigen_decomp_proj_to_pd`.

    Args:
        k (int): Dimension of the covariance matrices.
        proportion_var_dif (float): Probability of keeping an entry different between
            the 'bad' and 'good' matrices.
        generator (torch.Generator): Random number generator for reproducibility.
        var_range (Tuple[float, float], optional): Range for diagonal variances.
            Defaults to (0.0, 1.0).
        eps (float, optional): Small diagonal perturbation to ensure positive definiteness.
            Defaults to ``1e-6``.
        device (torch.device, optional): Device for tensor allocation. Defaults to CPU.
        dtype (torch.dtype, optional): Data type of the returned tensors. Defaults to ``torch.float64``.

    Returns:
        Tuple[torch.Tensor, torch.Tensor]:
            - ``sigma_bad``: Perturbed covariance matrix of shape ``(k, k)``, projected to PSD.
            - ``sigma_good``: Baseline covariance matrix of shape ``(k, k)``.

    Example:
        >>> g = torch.Generator().manual_seed(123)
        >>> sigma_bad, sigma_good = generate_sigma_bad_and_good(3, 0.5, generator=g)
        >>> sigma_bad.shape, sigma_good.shape
        (torch.Size([3, 3]), torch.Size([3, 3]))
    """
    # Step 1: Generate baseline matrices
    sigma_bad = random_vcov_matrix(k, generator=generator, var_range=var_range, device=device, dtype=dtype, eps=eps)
    sigma_good = random_vcov_matrix(k, generator=generator, var_range=var_range, device=device, dtype=dtype, eps=eps)

    # Step 2: Random mask for off-diagonal entries
    count_possible_changes = (k**2 + k) // 2 #Count diagonal entries + upper triangle
    index_change_vars = ~torch.bernoulli(torch.full((count_possible_changes,), proportion_var_dif, device=device), generator=generator).bool()

    triu_indices = torch.triu_indices(k, k, offset=0)
    indices_to_copy_sigma_bad = (triu_indices[0][index_change_vars], triu_indices[1][index_change_vars])

    sigma_good[indices_to_copy_sigma_bad] = sigma_bad[indices_to_copy_sigma_bad]
    i, j = torch.tril_indices(k, k, offset=-1)
    sigma_good[i, j] = sigma_good[j, i] # ensure symmetry
    
    sigma_good = eigen_decomp_proj_to_pd(sigma_good, eps=eps)

    return sigma_bad, sigma_good

def mvn_random_sample(
        mean : torch.Tensor, 
        cov_chol_decomp : Optional[torch.Tensor],
        n : int, 
        rng : Optional[torch.Generator] = None, 
        args_checks : bool = True
    ):
    """Generate n-vectors sampled of a multivariate normal (MVN) distribution with parameters
    mean and cov. Based on the implementation of (r)sample from 
    torch.distributions.MultivariateNormal according to torch version 2.9.1. It uses
    cholesky-decomposition method.

    Args:
        mean (torch.Tensor): Location parameter of a MVN. Shape ``(k,)`` or ``(b, k)`` or ``(1,5)``.
        cov_chol_decomp (torch.Tensor): Variance-Covariance matrix of MVN after cholesky decomposition. 
            Shape ``(k,k)``` or ``(b, k, k)`` or ``(1, k, k)``, ``cov.dim()==mean.dim()+1`` should hold.
        n (int): Count of vectors to be sampled (per batch).
        rng (Optional[torch.Generator]): If passed, sampling is done using this
            generator.
        args_checks (bool): If true, it will be checked whether the shapes of mean and
            cov are as expected, whether symmetry (w. r. t. to the last wo dims for each beach)
            is given within the range of ``symmetry_rtol_atol`` for ``cov`` and type checks
            are done for ``n`` and ``rng``.`
        symmetry_rtol_atol (Tuple[float,float]): Corresponds to the (rtol, a_tol) parameters
            of ``torch.allclose``, passed as ``*args``, so ordering is important. Ignored if
            ``not args_checks``.
    Returns:
        torch.Tensor:
            A tensor of shape ``(n, k)`` or ``(n, b, k)`` containing the ``n`` sampled vectors (for each batch).

    Example:
        >>> count_covariates = 5
        >>> device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        >>> torch.set_default_dtype(torch.float32)
        >>> rng = torch.Generator(device)
        >>> mu = torch.zeros(count_covariates)
        >>> sigma_bad, _ = generate_sigma_bad_and_good(k = count_covariates, proportion_var_dif=1.0, generator = rng, device=device, dtype= torch.get_default_dtype())
        >>> sample = mvn_random_sample(mean=mu, cov=sigma_bad, n=100, rng=rng)
        >>> sample.shape
        torch.Size([100, 5])
    """
    if args_checks:
        #shape checks
        assert (mean.dim() in [1, 2, 3]) and (cov_chol_decomp.dim()==mean.dim()+1), "mean must be a single vector (rank 1 tensor) and cov a matrix (rank 2 tensor)"
        assert (mean.size(-1) == cov_chol_decomp.size(-1)) and (cov_chol_decomp.size(-1) == cov_chol_decomp.size(-2)), "mean must have shape [k] and cov shape [k, k]"
        #ensure n is an int
        n = int(n)
        assert isinstance(rng, torch.Generator) or rng is None, "rng needs to be None or a rng"

    shape = torch.Size([n]) + mean.shape
    
    eps = torch.empty(shape, dtype=mean.dtype, device = mean.device).normal_(generator=rng)
    cov_chol_decomp = torch.linalg.cholesky(cov) # occurs over the last dimension

    deviations = torch.matmul(cov_chol_decomp, eps.unsqueeze(-1)).squeeze(-1) # apply decomp to each sampled vector


    return mean + deviations


## Understanding `biasAwareSelfLearning`

The signature:

```r
biasAwareSelfLearning <- function(accepts,
                                  rejects,
                                  target           = 'BAD', 
                                  filtering_beta   = c(0, 1),
                                  weak_learner     = 'classif.logreg',
                                  strong_learner   = 'classif.logreg',
                                  holdout_percent  = 0.1,
                                  sampling_percent = 1,
                                  labeling_percent = 0.01, 
                                  multiplier       = 1, 
                                  max_iterations   = 3,
                                  early_stop       = F,
                                  silent           = F) 
```

The call
```r
# BASL parameters
  filtering_beta   <- c(0.01, 0.99)
  holdout_percent  <- 0.1
  labeling_percent <- 0.1
  sampling_percent <- 0.8
  multiplier       <- 2
  max_iterations   <- 5
  early_stop       <- T
  
  # label rejected cases
  rej_labels <- biasAwareSelfLearning(accepts          = current_accepts, 
                                      rejects          = current_rejects,
                                      target           = 'BAD', 
                                      filtering_beta   = filtering_beta,
                                      weak_learner     = 'classif.logreg',
                                      strong_learner   = 'classif.logreg',
                                      holdout_percent  = holdout_percent,
                                      labeling_percent = labeling_percent, 
                                      sampling_percent = sampling_percent,
                                      multiplier       = multiplier, 
                                      max_iterations   = max_iterations,
                                      early_stop       = early_stop,
                                      silent           = T)
```

There are a couple of parts in the algorithm:

0. **Preparation:** defines several variables. 
    * specially interesting is that it creates "holdout samples" from the accepts when `early_stop == T`
    * it also defines as `bm_metric` the auc.
1. **Reject Filtering** It occurs if `filtering_beta != c(0,1)` (as the call is done inside the acceptance loop). The filtering is done via the function `filteringStage`

In [ ]:
from sklearn.ensemble import IsolationForest
from typing import Callable
class BaseBASL:
    def __init__(
            self,
            filtering_beta : torch.Tensor,
            weak_learner : Callable[[torch.Tensor], torch.Tensor],   # Consider making abstract class Learners
            strong_learner : Callable[[torch.Tensor], torch.Tensor], # Consider making abstract class Learners,
            hold_out_percent : float,
            labeling_percent : float,
            multiplier : float,
            max_iterations :int,
            early_stop : bool,
            isolation_forest : IsolationForest
    ):
        pass

    def self_learn(
            self,
            features_accept : torch.Tensor,
            default_flags_accept : torch.Tensor,
            features_reject : torch.Tensor,
            silent : bool = True
    ) -> torch.Tensor:
        pass

    def filter_rejects(
            self,
            features_accept : torch.Tensor,
            features_reject : torch.Tensor,
            *args,
            **kwargs
    ) -> torch.Tensor:
        pass

    def bayesian_evaluation( #Next step to implement
            self,
            features_accept : torch.Tensor,
            default_flag_accept : torch.Tensor,
            features_reject : torch.Tensor,
            *args,
            **kwargs
    ) -> torch.Tensor:
        pass

### Understanding `filteringStage`

Signature:

```r
filteringStage <- function(accepts, 
                           rejects, 
                           target    = 'BAD', 
                           beta      = c(0, 1),
                           num_trees = 100)
```

Call:

```r
filter_idx <- filteringStage(accepts   = train, # ifelse(early_stop, accepts[-holdout_idx_accepts, ], accepts)
                             rejects   = test,  # ifelse(early_stop, rejects[-holdout_idx_rejects, ], rejects)
                             beta      = filtering_beta, # From loop: c(0.01,0.99)
                             num_trees = 100)
```

The algorithm is based on isolation forest, as developed by Zelanzy7 (`devtools::install_github("Zelazny7/isofor")`). Is based solely on the features.

```r
  # remove target
  accepts[target] <- NULL
  
  # fit isolation forest
  mod <- iForest(X = accepts, nt = num_trees)
  
  # predict probs
  p_anom <- predict(mod, rejects[, !(colnames(rejects) %in% target)])
  
  # filter rejects
  tmp_rejects <- ((p_anom >= quantile(p_anom, beta[1])) & 
                     (p_anom <= quantile(p_anom, beta[2])))
```

The python implementation can be found in `sci-kit learn`, an extended option is the [Extended Isolation Forest](https://arxiv.org/abs/1811.02141) ([repo](https://github.com/sahandha/eif)) which can also be directly installed

In [ ]:
from sklearn.ensemble import IsolationForest
import numpy as np

forest = IsolationForest(n_estimators=100, max_samples="auto", random_state=1807)
def filter(
    ifo: IsolationForest,
    lower_trim_quantile: float,
    upper_trim_quantile: float,
    features: np.ndarray,
    return_index: bool = True,
) -> Union[np.ndarray, np.ndarray]:
    """
    Filters observations using a two-sided trimming strategy based on
    Isolation Forest normality scores.

    The function fits an Isolation Forest model on the provided feature
    matrix and computes the negative anomaly scores via
    ``IsolationForest.score_samples``. These scores induce a relative
    normality (similarity) ranking.

    Observations are retained if their score lies within the central
    quantile interval defined by ``lower_trim_quantile`` and
    ``upper_trim_quantile``. Consequently, both highly anomalous
    observations (lower tail) and overly typical observations
    (upper tail) are removed.

    This procedure corresponds to a two-sided percentile-based filtering
    scheme as described in Kozodoi et al. (2025), "Fighting Sampling Bias".

    Args:
        ifo (IsolationForest):
            An unfit ``IsolationForest`` instance used to compute
            normality scores.
        lower_trim_quantile (float):
            Lower quantile boundary in the interval ``[0, 1]``.
            Observations with scores below this quantile are discarded.
        upper_trim_quantile (float):
            Upper quantile boundary in the interval ``[0, 1]``.
            Observations with scores above this quantile are discarded.
        features (np.ndarray):
            Feature matrix of shape ``(n_samples, n_features)``.
        return_index (bool, optional):
            If ``True``, return a boolean mask indicating retained
            observations. If ``False``, return the filtered feature
            matrix. Defaults to ``True``.

    Returns:
        np.ndarray:
            If ``return_index`` is ``True``, a boolean array of shape
            ``(n_samples,)`` indicating which observations are retained.
            Otherwise, a feature matrix containing only the retained
            observations.

    Raises:
        ValueError:
            If ``lower_trim_quantile`` or ``upper_trim_quantile`` are
            outside the interval ``[0, 1]`` or if
            ``lower_trim_quantile >= upper_trim_quantile``.
    """
    if not 0.0 <= lower_trim_quantile <= 1.0:
        raise ValueError("lower_trim_quantile must be in the interval [0, 1].")
    if not 0.0 <= upper_trim_quantile <= 1.0:
        raise ValueError("upper_trim_quantile must be in the interval [0, 1].")
    if lower_trim_quantile >= upper_trim_quantile:
        raise ValueError(
            "lower_trim_quantile must be strictly smaller than upper_trim_quantile."
        )

    ifo.fit(features)
    normality_scores = ifo.score_samples(features)

    lower_score_bound, upper_score_bound = np.quantile(
        normality_scores,
        [lower_trim_quantile, upper_trim_quantile],
    )

    keep_mask = (
        (lower_score_bound <= normality_scores)
        & (normality_scores <= upper_score_bound)
    )

    return keep_mask if return_index else features[keep_mask]
keep_mask = filter(forest, beta_bottom=0.01, beta_top=0.99, features=features.numpy())


## Developing acceptance loop

In [ ]:
def accept_based_on_top_percentent_of_arbitrary_var(
        features : torch.Tensor, 
        default_flag : torch.Tensor, 
        var_for_rule : int,
        top_percent : float,
        default_value : int = 1, # 1 or 0
        min_count_bads : int = 4
):
    if var_for_rule >= features.shape[1]:
        raise ValueError("var_for_rule outside of index")
    
    cutoff = torch.quantile(features[:, var_for_rule], 1 - top_percent)

    accepts = features[:, var_for_rule] >= cutoff


    count_defaults_within_accepts = (default_flag[accepts] == default_value).sum()
    if count_defaults_within_accepts < min_count_bads:
        defaults_still_selectable = lidx_defaults_non_accepted.sum()
        if defaults_still_selectable == 0:
            return accepts
        
        count_bads_to_still_achieve = min_count_bads - count_defaults_within_accepts
        var_for_rule_vals_of_rejected_defaults = features[lidx_defaults_non_accepted][:, var_for_rule]
        
        if defaults_still_selectable <= count_bads_to_still_achieve:
            var_for_rule_vals_of_rejected_defaults = features[lidx_defaults_non_accepted][:, var_for_rule]
            accept_rule_to_include_all_defaults = features[:, var_for_rule] >=  var_for_rule_vals_of_rejected_defaults.min()
            return accept_rule_to_include_all_defaults
        
        new_cutoff = torch.topk(var_for_rule_vals_of_rejected_defaults,k=count_bads_to_still_achieve, largest=True).values[-1]

        return features[:, var_for_rule] >= new_cutoff
    
    return accepts
        


accepts = accept_based_on_top_percentent_of_arbitrary_var(
    features,
    default_flag,
    var_for_rule=0,
    top_percent=top_percent
)
default_value = 1
min_count_bads : int = 4
var_for_rule : int = 0
count_defaults_within_accepts = (default_flag[accepts] == default_value).sum()

lidx_defaults_non_accepted = (~accepts) & (default_flag == default_value)
defaults_still_selectable = lidx_defaults_non_accepted.sum()


var_for_rule_vals_of_rejected_defaults = features[lidx_defaults_non_accepted][:, var_for_rule]
new_cutoff = torch.topk(count_bads_to_still_achieve, k = 3, largest=True).values[-1]

In [ ]:
import torch

from credit_data_simulation import CreditDataGenerator, CreditData, accept_based_on_top_percentent_of_arbitrary_var

torch.set_default_dtype(torch.float64)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

initial_seed = 1807

#Acceptance loop parametrization
init_sample = 100
sample_size = 100
holdout_sample = 3000
num_gens = 300
top_percent = 0.2

determinstic_mixture_weights = True

data_gen = CreditDataGenerator.init_with_internal_logic(
    count_covariates=2,
    mean_bad_diff=torch.tensor([1.0,2.0]),
    covars = {
        "bad" : torch.tensor([[1.0, 0.2], [0.2,1.0]]),
        "good" : torch.tensor([[1.0,-0.2], [-0.2,1.0]])
    },
    iid = False,
    mixture_weights=None,
    bad_ratio = 0.5,
    noise_var=0.0,
    device = device,
    seed_credit_data_gen=initial_seed,
    #dtype=torch.get_default_dtype()
)

# Initial population
data_gen.manual_seed(initial_seed)
features, default_flag = data_gen.sample(100)

accepts = accept_based_on_top_percentent_of_arbitrary_var(
    features, 
    default_flag,
    var_for_rule=0,
    top_percent=top_percent,
    default_value = CreditDataGenerator.bad_good_encoding["bad"]
)

credit_data = CreditData(features, default_flag, accepts)

# Holdout Population
data_gen.manual_seed(-initial_seed)
holdout_features, holdout_flag = data_gen.sample(n=holdout_sample)
holdout_data = CreditData(
    holdout_features, holdout_flag, 
    accepted_initial=torch.ones(holdout_flag.shape, dtype=torch.bool) # All are "accepted"
)

# Acceptance Loop

stats = []

for gen_nr in range(1, num_gens + 1):
    if gen_nr % 10 == 0:
        print("-- Iteration", f"{gen_nr}/{num_gens}:", credit_data.accepted_count, 
              "accepts and", credit_data.rejected_count, " rejects")
        
    ## Gather current statistics
    current_stats : dict = credit_data.data_stats()


    data_gen.manual_seed(initial_seed + gen_nr)
    features, default_flag = data_gen.sample(100)

    




In [10]:
type(features.numel())

int

In [6]:
3/float("nan")

nan

In [3]:
data_gen.rng.manual_seed(1807)
features, default_flag = data_gen.sample(100)


data_gen.rng.manual_seed(1807)
again_features, again_default = data_gen.sample(100)

(features == again_features).all()

tensor(True)

In [7]:
import pandas as pd

adaptations = pd.DataFrame(
    [
        ("combo_bad_ratio", "bad_ratio"),
        ("combo_count", "n"),
        ("combo_n1", "n_bad"),
        ("combo_n2", "n_good"),
        ("k_con", "count_covariates"),
        ("con_mean_bad_dif", "mean_bad_dif")
    ],
    columns = ["Old", "New"]
)
adaptations

,Old,New
0,combo_bad_ratio,bad_ratio
1,combo_count,n
2,combo_n1,n_bad
3,combo_n2,n_good
4,k_con,count_covariates
5,con_mean_bad_dif,mean_bad_dif


In [97]:
%run "../BASL/classifiers.py"

In [113]:
import numpy as np

import statsmodels.api as sm
from statsmodels.genmod import families

from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression
import time

import warnings
warnings.filterwarnings("ignore", message="Setting penalty=None will ignore the C and l1_ratio parameters")

iris = load_iris()
X = iris.data          # shape (150, 4)
X_torch = torch.from_numpy(X)
y = iris.target.astype(float)        # shape (150,)
feature_names = iris.feature_names
target_names = iris.target_names

wished_target = "versicolor"


y_bin = (y == np.where(target_names==wished_target)[0].item()).astype(float)

for label, desc in [(y_bin, "binary"), (y, "multiclass")]:
    is_binary = desc == "binary"
    label_torch = torch.from_numpy(label)
    label_torch = label_torch.unsqueeze(-1) if is_binary else label_torch.to(int)
    print("\nBeginning", desc, "comparision:")

    print("\tStep 0: Estimation")
    if is_binary: # This gives an error
        print("\t\t0. GLM:")
        begin_glm = time.time()
        X_glm = sm.add_constant(X)
        glm_lr = sm.GLM(
            endog = label,
            exog = X_glm,
            family = families.Binomial()
        )
        fitted_glm = glm_lr.fit()
        end_glm = time.time()
        print("\t\t\tTime needed:", round((end_sklearn - begin_sklearn)*1e3,2), "(ns)")

    print("\t\t1. scikit learn")
    begin_sklearn = time.time()
    sk_lr = LogisticRegression(C=np.inf, l1_ratio=0, solver="lbfgs")
    sk_lr = sk_lr.fit(X, label)
    end_sklearn = time.time()
    print("\t\t\tTime needed:", round((end_sklearn - begin_sklearn)*1e3,2), "(ns)")

    print("\t\t2. torch implementation")
    begin_torch = time.time()
    torch_lr = TorchLogistic(n_features=X.shape[1], n_classes=label_torch.max()+1)
    torch_lr.fit(X_torch, label_torch, reduction='sum')
    end_torch = time.time()
    print("\t\t\tTime needed:", round((end_torch - begin_torch)*1e3, 2), "(ms)")

    print("\tStep 1: Parameter Comparision. Order (GLM), Sklearn, torch")
    print("\t\tComparision of weights")
    print(torch.cat(
        ([torch.from_numpy(fitted_glm.params[1:]).unsqueeze(0)] if is_binary else []) + [
            torch.from_numpy(sk_lr.coef_), torch_lr.lin_estimator.weight.detach()
            ]
    ))

    print("\n\t\tComparision of intercept")
    print(torch.stack(
        ([torch.from_numpy(fitted_glm.params[[0]])] if is_binary else []) + [
            torch.from_numpy(sk_lr.intercept_), torch_lr.lin_estimator.bias.detach()
        ]
        ))


    print("\tStep 2: Comparision of predicted probabilities.")
    sk_probs = torch.from_numpy(sk_lr.predict_proba(X))
    torch_probs = torch_lr.predict_proba(X_torch).detach()

    # This has to bee completed with the 
    print("\t\tRandomly chosen probs:")
    print(torch.stack([sk_probs, torch_probs.detach()], dim=1)[torch.randint(0, sk_probs.shape[0], size=(6,))])
    print("\t\tAbsolute Distances: (mean and quantiles=.01,.25,.5,.75,.99)")
    mae = lambda a, b : (a-b).abs().mean().detach().item()
    ae_quantiles = lambda a, b, q=torch.tensor([0.01,0.25,0.5,.75,.99]): (a - b).abs().quantile(q)
    print("\t\t\tscikit vs. torch:")
    for f in [mae, ae_quantiles]:
        print("\t\t\t", f(sk_probs, torch_probs))

    if is_binary:
        glm_probs = torch.from_numpy(fitted_glm.predict(X_glm))
        glm_probs = torch.stack([1 - glm_probs, glm_probs], dim=1)
        
        print("\t\tGLM vs sklearn:")
        print(mae(glm_probs, sk_probs))
        print(ae_quantiles(glm_probs, sk_probs))

        print("\t\tGLM vs torch:")
        print(mae(glm_probs, torch_probs))
        print(ae_quantiles(glm_probs, torch_probs))
    




Beginning binary comparision:
	Step 0: Estimation
		0. GLM:
			Time needed: 8.09 (ns)
		1. scikit learn
			Time needed: 70.66 (ns)
		2. torch implementation
Binary
			Time needed: 218.63 (ms)
	Step 1: Parameter Comparision. Order (GLM), Sklearn, torch
		Comparision of weights
tensor([[-0.2454, -2.7966,  1.3136, -2.7783],
        [-0.2438, -2.7944,  1.3121, -2.7756],
        [-0.2454, -2.7966,  1.3136, -2.7783]])

		Comparision of intercept
tensor([[7.3785],
        [7.3655],
        [7.3785]])
	Step 2: Comparision of predicted probabilities.
		Randomly chosen probs:
tensor([[[0.5148, 0.4852],
         [0.5148, 0.4852]],

        [[0.9958, 0.0042],
         [0.9958, 0.0042]],

        [[0.8015, 0.1985],
         [0.8017, 0.1983]],

        [[0.2911, 0.7089],
         [0.2908, 0.7092]],

        [[0.4083, 0.5917],
         [0.4078, 0.5922]],

        [[0.9212, 0.0788],
         [0.9213, 0.0787]]])
		Absolute Distances: (mean and quantiles=.01,.25,.5,.75,.99)
			scikit vs. torch:
			 0.0

In [112]:
torch.from_numpy(fitted_glm.params[[0]])

tensor([7.3785])

In [86]:
label_torch.max()+1

tensor(3.)

In [83]:
sk_lr.intercept_

array([  3.97232109,  19.28077884, -23.25309993])

In [84]:
torch_lr.lin_estimator.bias.detach()

tensor([-1.0898e+49])

In [ ]:
mae = lambda a, b : (a-b).abs().mean().detach().item()
mae_quantiles = lambda a, b, q=torch.tensor([0.01,0.25,0.5,.75,.99]): (a - b).abs().quantile(q)
mae(sk_probs, torch_probs)

0.0001512147756215606

In [ ]:
q = torch.tensor([0.01,0.25,0.5,.75,.99])
(a - b).abs().quantile(q)

tensor([5.1907e-06, 6.2098e-05, 1.1801e-04, 2.1455e-04, 4.7902e-04])

In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
import warnings
warnings.filterwarnings("ignore", message="Setting penalty=None will ignore the C and l1_ratio parameters")

lr = LogisticRegression(C=np.inf, l1_ratio=0, solver="lbfgs")
lr = lr.fit(X, y.astype(float))

probs = lr.predict_proba(X)
roc_auc_score(y, probs[:,1])

0.8258

In [ ]:
%run "../BASL/classifiers.py"
torch.set_default_dtype(torch.float64)

t_lr = TorchLogistic(n_features=X.shape[1], n_classes=2)
t_lr.fit(torch.from_numpy(X), torch.from_numpy(y.astype(float)).unsqueeze(-1), reduction='sum')
probs_torch = t_lr.predict_proba(torch.from_numpy(X))

In [25]:
print("Comparision of weights (torch first, then scikit learn)")
print(torch.cat([t_lr.lin_estimator.weight.detach(), torch.from_numpy(lr.coef_)]))

print("\nComparision of intercept (torch first, then scikit learn)")
print(torch.stack([t_lr.lin_estimator.bias.detach(), torch.from_numpy(lr.intercept_)]))

Comparision of weights (torch first, then scikit learn)
tensor([[-0.2454, -2.7966,  1.3136, -2.7783],
        [-0.2438, -2.7944,  1.3121, -2.7756]])

Comparision of intercept (torch first, then scikit learn)
tensor([[7.3785],
        [7.3655]])


tensor([[7.3785],
        [7.3655]])

In [36]:
probs - probs_torch.detach().numpy()

array([[-7.35099597e-05,  7.35099597e-05],
       [ 9.13539872e-05, -9.13539872e-05],
       [ 2.48551023e-05, -2.48551023e-05],
       [ 1.66740009e-04, -1.66740009e-04],
       [-6.30880567e-05,  6.30880567e-05],
       [-5.42004837e-05,  5.42004837e-05],
       [-1.93725757e-05,  1.93725757e-05],
       [-4.62795064e-05,  4.62795064e-05],
       [ 3.36684888e-04, -3.36684888e-04],
       [ 1.40465540e-04, -1.40465540e-04],
       [-8.53785375e-05,  8.53785375e-05],
       [ 5.01153843e-06, -5.01153843e-06],
       [ 1.99080160e-04, -1.99080160e-04],
       [ 2.46223413e-04, -2.46223413e-04],
       [-4.93804185e-05,  4.93804185e-05],
       [-1.77461800e-05,  1.77461800e-05],
       [-4.10487150e-05,  4.10487150e-05],
       [-7.47528768e-05,  7.47528768e-05],
       [-8.45591844e-05,  8.45591844e-05],
       [-5.56194554e-05,  5.56194554e-05],
       [-9.14354195e-05,  9.14354195e-05],
       [-5.74957529e-05,  5.74957529e-05],
       [-4.25488687e-05,  4.25488687e-05],
       [-7.

array([[0.91508692, 0.08491308],
       [0.7170825 , 0.2829175 ],
       [0.82801758, 0.17198242],
       [0.73198568, 0.26801432],
       [0.93292491, 0.06707509],
       [0.9765968 , 0.0234032 ],
       [0.90489952, 0.09510048],
       [0.87455348, 0.12544652],
       [0.62894652, 0.37105348],
       [0.69007998, 0.30992002],
       [0.94679586, 0.05320414],
       [0.85338465, 0.14661535],
       [0.65195987, 0.34804013],
       [0.7107604 , 0.2892396 ],
       [0.98537304, 0.01462696],
       [0.99578894, 0.00421106],
       [0.98602841, 0.01397159],
       [0.93433233, 0.06566767],
       [0.96257658, 0.03742342],
       [0.96652251, 0.03347749],
       [0.85535671, 0.14464329],
       [0.9664635 , 0.0335365 ],
       [0.95520489, 0.04479511],
       [0.90529445, 0.09470555],
       [0.79694428, 0.20305572],
       [0.66637663, 0.33362337],
       [0.91420848, 0.08579152],
       [0.90640949, 0.09359051],
       [0.89304893, 0.10695107],
       [0.76450675, 0.23549325],
       [0.

In [43]:
from typing import Union
Union[None, str]

typing.Optional[str]

In [1]:
%run "../BASL/classifiers.py"